# Dinamika Glukosa–Insulin

**ID proyek:** `O005-LEGA-V101-PRJ05`  
**Status:** titik awal pedagogis yang ditulis secara independen.

Notebook ini menggunakan data sintetis/terbuka saja. Notebook ini **bukan** kode atau data dari makalah yang dikutip dalam bab sumber dan **bukan** klaim reproduksi hasil penelitian mana pun.


## Pertanyaan pemodelan

Parameter mana yang mengatur tinggi puncak glukosa dan waktu pulih setelah masukan makanan sintetis?

Tujuan kerja: tetapkan sistem, jalankan eksperimen deterministik, periksa invarian, visualisasikan perilaku, lalu kritik kecukupan model.


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

SEED = 2026082205
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)


## Struktur dan asumsi

Tiga keadaan—glukosa, aksi insulin, dan insulin—mengikuti ODE minimal; masukan makanan berupa pulsa Gaussian yang diketahui; keadaan awal berada pada basal.

Semua skala dan parameter di notebook ini bersifat ilustratif. Ubah satu asumsi pada satu waktu dan catat dampaknya pada keluaran serta invarian.


In [ ]:
G_b, I_b = 90.0, 10.0
t_eval = np.linspace(0.0, 240.0, 961)

def meal_input(t):
    return 2.8 * np.exp(-0.5 * ((t - 25.0) / 10.0) ** 2)

def glucose_rhs(t, y):
    G, X, I = y
    dG = -0.025 * (G - G_b) - X * G + meal_input(t)
    dX = -0.080 * X + 0.00012 * (I - I_b)
    dI = -0.120 * (I - I_b) + 0.40 * max(G - G_b, 0.0)
    return [dG, dX, dI]

glucose_run = solve_ivp(glucose_rhs, (0.0, 240.0), [G_b, 0.0, I_b], t_eval=t_eval, rtol=1e-9, atol=1e-10)
G, X, insulin = glucose_run.y
sample_minutes = np.arange(0, 241, 15)
synthetic_glucose = np.interp(sample_minutes, t_eval, G) + rng.normal(0.0, 1.2, sample_minutes.size)


## Pemeriksaan numerik

Pemeriksaan berikut sengaja berada di dalam notebook: eksekusi berhenti bila suatu invarian dasar gagal. Ini bukan bukti bahwa model benar; ini hanya bukti bahwa implementasi memenuhi kontrak numerik terbatasnya.


In [ ]:
assert glucose_run.success and np.isfinite(glucose_run.y).all()
assert float(G.max()) > G_b + 8.0
assert abs(float(G[-1]) - G_b) < 2.0
assert np.min(G) > 0.0 and np.min(insulin) > 0.0


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
axes[0].plot(t_eval, G, label="glukosa model")
axes[0].scatter(sample_minutes, synthetic_glucose, s=16, color="black", label="data sintetis")
axes[0].axhline(G_b, color="gray", linestyle="--", linewidth=1)
axes[0].set(ylabel="glukosa (satuan ilustratif)", title="Respons terhadap masukan makanan")
axes[0].legend(fontsize=8)
axes[1].plot(t_eval, insulin, color="tab:orange", label="insulin")
axes[1].plot(t_eval, 10 * meal_input(t_eval), color="tab:green", alpha=0.7, label="10 × masukan")
axes[1].set(xlabel="menit", ylabel="satuan ilustratif")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()
plt.close(fig)


## Validasi, identifikasi, dan keterbatasan

Keterbatasan awal: Model bukan alat diagnosis dan mengabaikan variasi organ, hormon lain, ketidakpastian makanan, serta perbedaan pasien.

Jawab sebelum menafsirkan gambar:

1. Besaran apa yang benar-benar dapat diamati, dan bagaimana galat pengukurannya dimodelkan?
2. Parameter mana yang dapat diidentifikasi dari keluaran tersebut? Tunjukkan dengan profil galat, pemisahan latih/uji, atau eksperimen sensitivitas.
3. Invarian atau pola kualitatif apa yang harus tetap benar ketika ukuran langkah, benih acak, atau resolusi diubah?
4. Temukan satu skenario kegagalan model dan jelaskan data tambahan yang diperlukan untuk membedakannya dari model alternatif.


## Daftar periksa reproduksibilitas

- [ ] Gunakan CPython dan versi paket tepat seperti `requirements.lock`.
- [ ] Jalankan ulang dari kernel kosong tanpa jaringan.
- [ ] Pertahankan nilai `SEED` (benih acak), lalu ulangi dengan sedikitnya lima benih acak lain dan laporkan variasinya.
- [ ] Catat setiap perubahan parameter, persamaan, toleransi, serta pembagian data.
- [ ] Pastikan semua uji lulus dan jelaskan mengapa tiap uji relevan.
- [ ] Simpan hasil turunan di luar notebook sumber; notebook distribusi harus tetap tanpa keluaran tersimpan.
- [ ] Bedakan hasil simulasi, data sintetis, dan klaim empiris secara eksplisit.
